# Unlimited-OCR on the scanned bank statements (Colab / Kaggle GPU)

Runs `baidu/Unlimited-OCR` over the scan PDFs and saves **plain-text output per document** so
`python -m stmtai.benchmark score` can compare it with RapidOCR and Tesseract on the same files.

**Before you start:** Runtime > Change runtime type > a GPU (T4 is enough to try; the model is 3B parameters in bf16,
so if it runs out of memory, restart and use `PAGES_PER_CALL = 1`).
Upload `scans.zip` (the 32 `*__scan_*.pdf` files from `data/synthetic`) via the Files panel.

**Tested once by the author on a T4 (about 1.5 hours for all 32 scans).** The model card does not document its output files, so the loader below captures the
return value of `infer_multi` and captures what it prints. The full 32-document run takes about 1.5 hours on a free T4.

In [ ]:
!pip -q install transformers==4.57.1 einops addict easydict pymupdf matplotlib
!unzip -oq scans.zip -d scans
!ls scans | head

In [ ]:
import torch, glob, os, re, json, pathlib, fitz
from transformers import AutoModel, AutoTokenizer

NAME = 'baidu/Unlimited-OCR'
tok = AutoTokenizer.from_pretrained(NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(NAME, trust_remote_code=True, use_safetensors=True,
                                  torch_dtype=torch.bfloat16).eval().cuda()
print('loaded', torch.cuda.get_device_name(0))

In [ ]:
OUT = pathlib.Path('bench/unlimited-ocr'); OUT.mkdir(parents=True, exist_ok=True)
RAW = pathlib.Path('bench_raw'); RAW.mkdir(exist_ok=True)
PAGES_PER_CALL = 8   # statements are 1-3 pages; lower this if you hit out-of-memory
DET = re.compile(r'<\|det\|>.*?<\|/det\|>', re.S)   # layout markers: keep the text, drop the boxes

def render(pdf, dpi=300):
    paths = []
    for i, page in enumerate(fitz.open(pdf)):
        p = f'tmp/{pathlib.Path(pdf).stem}_p{i}.png'
        os.makedirs('tmp', exist_ok=True)
        page.get_pixmap(dpi=dpi).save(p); paths.append(p)
    return paths

MARK = '===============save results:==============='

def run_one(pdf):
    # The model prints its answer; its own save step silently writes nothing for some outputs
    # (those not starting with <PAGE>), so capture stdout instead of reading saved files.
    pages = render(pdf)
    text = ''
    for i in range(0, len(pages), PAGES_PER_CALL):
        out_dir = RAW / pathlib.Path(pdf).stem
        out_dir.mkdir(exist_ok=True)
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            res = model.infer_multi(tok, prompt='<image>Multi page parsing.', image_files=pages[i:i+PAGES_PER_CALL],
                                    output_path=str(out_dir), image_size=1024, max_length=32768,
                                    no_repeat_ngram_size=35, ngram_window=1024, save_results=True)
        captured = buf.getvalue().split(MARK)[0]
        text += (res if isinstance(res, str) and len(res) > len(captured) else captured) + '\n'
    return text

for pdf in sorted(glob.glob('scans/*.pdf')):
    dest = OUT / (pathlib.Path(pdf).stem + '.txt')
    if dest.exists():
        continue
    raw = run_one(pdf)
    (RAW / (pathlib.Path(pdf).stem + '.raw.txt')).write_text(raw)
    dest.write_text(DET.sub('', raw))
    print('done', dest.name, len(raw), 'chars', flush=True)

In [ ]:
# A quick look before downloading: tables usually come back as HTML or markdown; the benchmark only needs the numbers.
print(next(OUT.glob('*scan_light*.txt')).read_text()[:1500])

In [ ]:
!zip -qr unlimited-ocr-output.zip bench/unlimited-ocr bench_raw
# Download unlimited-ocr-output.zip, unzip it so the files sit in data/bench/unlimited-ocr/ in the repo, then run:
#   python -m stmtai.benchmark score --out reports